# Independent Work: Experiments with Classification, Regression and Clustering

Independent work for the second half of the lab – it is not graded. Work on your own or in pairs and ask when you get stuck.

All cells run as they are. The tasks say what to **change** in the code (lines marked `# CHANGE`); after a change, run the cell again and see how the result changed.
We will discuss the questions at the end together.

We do **not** preprocess the data (scaling etc.) – that is the topic of lab 08.

## 1. Data preparation – Wine

Chemical analysis of 178 wines from three cultivars (classes 0, 1, 2), 13 numerical features.

We split the data into a **training** (75 %) and a **validation** (25 %) part. Today we do not make any "final" choice, so we do not need a test set –
model selection followed by a final test comes in lab 07.

In [ ]:
import numpy as np                     # arrays and maths
import pandas as pd                    # tables
import matplotlib.pyplot as plt        # plots
from IPython.display import display    # displaying tables

from sklearn import datasets, metrics
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

RSTATE = 42                                                       # reproducible results

wine = datasets.load_wine(as_frame=True)
X, y = wine.data, wine.target                                     # features (DataFrame) and cultivar (Series)
X_train, X_val, y_train, y_val = train_test_split(                # one joint split of X and y
    X, y, test_size=0.25, stratify=y, random_state=RSTATE
)
df = X_train.assign(target=y_train)                               # table for browsing: training data + target
print("Train:", X_train.shape, " Validation:", X_val.shape)

## 2. Browsing the data in pandas

**Task A – selecting rows and columns**
1. Change `ALCOHOL_LIMIT` from 14 to 13.
2. Add the feature `"flavanoids"` to `COLUMNS`.
3. Extend the mask with the condition `df["proline"] > 800` – first using `&`, then using `|`. How does the number of rows change?
4. Sort the displayed selection by `proline` in descending order: `.sort_values("proline", ascending=False)`.

In [ ]:
ALCOHOL_LIMIT = 14                                      # CHANGE: alcohol content threshold
COLUMNS = ["alcohol", "proline", "target"]              # CHANGE: displayed columns
mask = df["alcohol"] > ALCOHOL_LIMIT                    # CHANGE: add a condition on proline
selection = df.loc[mask, COLUMNS]                       # rows by mask, selected columns
print("Selected rows:", mask.sum())
display(selection.head(8))                              # CHANGE: add sorting before .head()

**Task B – summaries**
1. Also display the class **shares** using `value_counts(normalize=True)`.
2. Sort the table of ranges by the column `range` in descending order. Which feature has by far the largest range?

In [ ]:
display(y_train.value_counts().sort_index().rename("count"))     # number of wines of each cultivar
# CHANGE: add the class shares here

ranges = X_train.agg(["min", "max"]).T                           # minimum and maximum of each feature
ranges["range"] = ranges["max"] - ranges["min"]                  # new column: max - min
display(ranges.round(2))                                         # CHANGE: sort by range

## 3. Classification – features and model complexity

**Task C – decision tree experiment**
1. Add the set `"alcohol_only": ["alcohol"]` to `FEATURE_SETS`.
2. Extend `DEPTHS` to all values 1–10 (`list(range(1, 11))`).
3. Run both following cells. Which feature set and depth would you guess are best? Is the model overfitting anywhere?

All variants use the same training and validation split. The `target` column does not belong to any feature set.

In [ ]:
FEATURE_SETS = {                                         # CHANGE: set name -> list of features
    "all": list(X_train.columns),                        # all 13 measurements
    "two": ["alcohol", "flavanoids"],                    # a smaller set for comparison
}
DEPTHS = [1, 2, 3, 5]                                    # CHANGE: maximal depths to try

baseline_acc = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).score(X_val, y_val)
rows = []
for set_name, columns in FEATURE_SETS.items():           # every feature set ...
    for depth in DEPTHS:                                 # ... with every depth
        tree = DecisionTreeClassifier(max_depth=depth, random_state=RSTATE)
        tree.fit(X_train[columns], y_train)              # learn only on the selected columns
        rows.append({"set": set_name, "depth": depth,
                     "train_accuracy": tree.score(X_train[columns], y_train),
                     "val_accuracy": tree.score(X_val[columns], y_val)})
results = pd.DataFrame(rows)
display(results.round(3))
print("Baseline validation accuracy:", round(baseline_acc, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for set_name, group in results.groupby("set"):           # one curve per feature set
    ax.plot(group["depth"], group["val_accuracy"], "o-", label=set_name)
ax.axhline(baseline_acc, color="gray", linestyle="--", label="baseline")   # reference level
ax.set(xlabel="max_depth", ylabel="validation accuracy", ylim=(0, 1.05))
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 4. Regression – Diabetes

**Task D – numerical predictions**
1. Extend `REG_DEPTHS` to `[1, 2, 3, 5, 10]`.
2. Add the set `"bmi": ["bmi"]` to `REG_SETS`.
3. Sort the results table by `val_MAE` in ascending order. Which model has the smallest error? Which one is only slightly better than the mean?

In [ ]:
diabetes = datasets.load_diabetes(as_frame=True)
Xr_train, Xr_val, yr_train, yr_val = train_test_split(           # training and validation part
    diabetes.data, diabetes.target, test_size=0.25, random_state=RSTATE
)
REG_DEPTHS = [1, 3]                                              # CHANGE: depths of regression trees
REG_SETS = {"all": list(Xr_train.columns)}                       # CHANGE: add a set with bmi only

rows = []
for set_name, columns in REG_SETS.items():
    models = [("baseline (mean)", DummyRegressor(strategy="mean")), ("linear", LinearRegression())]
    models += [(f"tree {d}", DecisionTreeRegressor(max_depth=d, random_state=RSTATE)) for d in REG_DEPTHS]
    for name, model in models:
        model.fit(Xr_train[columns], yr_train)                   # learn on training data
        pred = model.predict(Xr_val[columns])                    # predictions for validation data
        rows.append({"set": set_name, "model": name,
                     "train_MAE": metrics.mean_absolute_error(yr_train, model.predict(Xr_train[columns])),
                     "val_MAE": metrics.mean_absolute_error(yr_val, pred),
                     "val_RMSE": metrics.root_mean_squared_error(yr_val, pred)})
reg_results = pd.DataFrame(rows)
display(reg_results.round(1))                                    # CHANGE: sort by val_MAE

## 5. Clustering – Iris

**Task E – groups without known targets**
1. Extend `K_VALUES` to all integers 2–8.
2. Add the set `"petal": ["petal length (cm)", "petal width (cm)"]` to `CLUSTER_SETS`.
3. Add a column with the size of the smallest cluster to the table: `"smallest": pd.Series(labels).value_counts().min()`.

In [ ]:
iris = datasets.load_iris(as_frame=True)
X_iris = iris.data                                               # measurements only, K-means gets no species
K_VALUES = [2, 3, 5]                                             # CHANGE: numbers of clusters 2–8
CLUSTER_SETS = {"all": list(X_iris.columns)}                     # CHANGE: add the petal set

rows = []
for set_name, columns in CLUSTER_SETS.items():
    for k in K_VALUES:
        km = KMeans(n_clusters=k, n_init=10, random_state=RSTATE)
        labels = km.fit_predict(X_iris[columns])                 # learning without the target
        rows.append({"set": set_name, "k": k,                    # CHANGE: add the size of the smallest cluster
                     "silhouette": metrics.silhouette_score(X_iris[columns], labels)})
cluster_results = pd.DataFrame(rows)
display(cluster_results.round(3))

ax = cluster_results.pivot(index="k", columns="set", values="silhouette").plot(marker="o", figsize=(8, 4))
ax.set(xlabel="number of clusters k", ylabel="silhouette"); ax.grid(alpha=0.3)
plt.show()

**Task F – clustering wines**

Run K-means with 3 clusters on the wines (`X`) and compare the clusters with the true cultivars using the Adjusted Rand Index. The trees had a high validation accuracy – why does clustering match the cultivars so much worse?
Look again at the table of ranges from Task B. (We will fix this in lab 08.)

In [ ]:
km_wine = KMeans(n_clusters=3, n_init=10, random_state=RSTATE)
labels_wine = km_wine.fit_predict(X)                             # clusters of wines without cultivars
print("Wine ARI:", round(metrics.adjusted_rand_score(y, labels_wine), 3))
# CHANGE (optional): try clustering by the "proline" column only – how does the ARI change?

## Questions for the joint discussion

1. How did the selection of rows change when using `&` and `|`?
2. Which wine feature has the largest range? Does that make it the most useful one?
3. How did the validation accuracy change with the feature set? Is the single feature `alcohol` enough?
4. For which depths is the tree underfitting? Why do we not decide by the training accuracy?
5. How do regression metrics (MAE, RMSE) differ from classification accuracy? Why can RMSE come out larger than MAE?
6. How did the model with `bmi` only do compared to the baseline and to all features?
7. Does the highest silhouette agree with the true number of iris species (3)?
8. Why does clustering the wines disagree so much with the cultivars?

[Back to the main notebook](ML_Introduction_EN.ipynb)